# Detecção de Buracos com CNN Baseline

Notebook para treino e avaliacao de uma CNN baseline usando o dataset local `datasets/whole-detection/archive`.

**O que e uma CNN Baseline:** e uma rede convolucional simples usada como ponto de partida para classificacao de imagens, servindo como referencia inicial de desempenho antes de modelos mais complexos.

## 1. Objetivo

- Carregar o dataset local de imagens de pista normal e com buracos.
- Treinar uma CNN baseline para classificação binária.
- Avaliar com acurácia, precision, recall, f1-score e matriz de confusão.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Localiza a raiz do projeto para importar o pacote src de forma robusta no VS Code.
raiz_atual = Path.cwd().resolve()
projeto_raiz = next((p for p in [raiz_atual, *raiz_atual.parents] if (p / "src").exists()), raiz_atual)

if str(projeto_raiz) not in sys.path:
    sys.path.insert(0, str(projeto_raiz))

from src.deteccao_buracos import (
    MAPA_CLASSES,
    PROPORCAO_VALIDACAO_PADRAO,
    SEMENTE_PADRAO,
    TAMANHO_IMAGEM_PADRAO,
    avaliar_modelo,
    carregar_imagens_rotuladas,
    construir_cnn_baseline,
    obter_caminhos_dataset,
    separar_treino_validacao,
    treinar_modelo,
)

sns.set_theme(style="whitegrid")
print(f"Raiz do projeto: {projeto_raiz}")

## 2. Configuração

Definimos parametros de reproducibilidade, resolucao de imagem e hiperparametros de treino.

In [ ]:
epocas = 10
batch_size = 32
forma_entrada = (TAMANHO_IMAGEM_PADRAO[0], TAMANHO_IMAGEM_PADRAO[1], 3)

caminhos = obter_caminhos_dataset()
print("Caminhos do dataset:")
for chave, valor in caminhos.items():
    print(f"- {chave}: {valor}")

print(f"\nMapa de classes: {MAPA_CLASSES}")
print(f"Tamanho de imagem: {TAMANHO_IMAGEM_PADRAO}")
print(f"Semente: {SEMENTE_PADRAO}")

## 3. Carregamento e pré-processamento

As imagens são carregadas já normalizadas no intervalo `[0, 1]` e com rótulos binários.

In [ ]:
X, y = carregar_imagens_rotuladas(caminhos["base"], TAMANHO_IMAGEM_PADRAO)

print(f"Shape de X: {X.shape}")
print(f"Shape de y: {y.shape}")
print(f"Faixa de valores em X: min={X.min():.4f}, max={X.max():.4f}")

classes_unicas, contagens = np.unique(y, return_counts=True)
distribuicao = dict(zip(classes_unicas, contagens))
print(f"Distribuicao de classes: {distribuicao}")

## 4. EDA básica

Visualizamos distribuição de classes e amostras de cada categoria.

In [ ]:
labels_legiveis = ["normal", "buracos"]
valores = [int(np.sum(y == MAPA_CLASSES["normal"])), int(np.sum(y == MAPA_CLASSES["buracos"]))]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].pie(valores, labels=labels_legiveis, autopct="%1.1f%%", colors=["#2e8b57", "#cc3333"])
axes[0].set_title("Distribuicao de classes")

sns.barplot(x=labels_legiveis, y=valores, ax=axes[1], palette=["#2e8b57", "#cc3333"])
axes[1].set_title("Contagem por classe")
axes[1].set_ylabel("Quantidade")
plt.tight_layout()
plt.show()

# Exibe 4 amostras de cada classe para inspecao visual rapida.
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
indices_normal = np.where(y == MAPA_CLASSES["normal"])[0][:4]
indices_buracos = np.where(y == MAPA_CLASSES["buracos"])[0][:4]

for i, idx in enumerate(indices_normal):
    axes[0, i].imshow(X[idx])
    axes[0, i].set_title("normal")
    axes[0, i].axis("off")

for i, idx in enumerate(indices_buracos):
    axes[1, i].imshow(X[idx])
    axes[1, i].set_title("buracos")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

## 5. Separação treino-validação

Separação estratificada para manter proporção de classes nos dois conjuntos.

In [ ]:
X_treino, X_valid, y_treino, y_valid = separar_treino_validacao(
    X, y, PROPORCAO_VALIDACAO_PADRAO, SEMENTE_PADRAO
)

print(f"X_treino: {X_treino.shape} | y_treino: {y_treino.shape}")
print(f"X_valid: {X_valid.shape} | y_valid: {y_valid.shape}")

## 6. Treino da CNN baseline

Modelo convolucional simples para classificação binária.

In [ ]:
modelo = construir_cnn_baseline(forma_entrada)
modelo.summary()

In [ ]:
historico = treinar_modelo(
    modelo=modelo,
    X_treino=X_treino,
    y_treino=y_treino,
    X_valid=X_valid,
    y_valid=y_valid,
    epocas=epocas,
    batch_size=batch_size,
)

## 7. Avaliação

Calculamos métricas e exibimos a matriz de confusão.

In [ ]:
resultados = avaliar_modelo(modelo, X_valid, y_valid)

print(f"Acuracia de validacao: {resultados['acuracia']:.4f}")
print("\nMatriz de confusao:")
print(resultados["matriz_confusao"])

print("\nRelatorio de classificacao (resumo):")
for classe in ["normal", "buracos"]:
    metricas = resultados["relatorio_classificacao"][classe]
    print(
        f"{classe:9s} | precision={metricas['precision']:.4f} "
        f"recall={metricas['recall']:.4f} f1-score={metricas['f1-score']:.4f}"
    )

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(
    resultados["matriz_confusao"],
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["normal", "buracos"],
    yticklabels=["normal", "buracos"],
)
plt.title("Matriz de confusao - CNN baseline")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(historico.history["loss"], label="Treino")
axes[0].plot(historico.history["val_loss"], label="Validacao")
axes[0].set_title("Loss por epoca")
axes[0].set_xlabel("Epoca")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(historico.history["accuracy"], label="Treino")
axes[1].plot(historico.history["val_accuracy"], label="Validacao")
axes[1].set_title("Acuracia por epoca")
axes[1].set_xlabel("Epoca")
axes[1].set_ylabel("Acuracia")
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Estimativas reais de beneficio (com base nos dados)

Abaixo usamos as **previsoes reais do conjunto de validacao** para estimar impacto operacional.

- `TP`: buracos corretamente detectados
- `FP`: alertas falsos (imagem normal marcada como buraco)
- `FN`: buracos perdidos
- `TN`: imagens normais corretamente ignoradas

As estimativas economicas sao apresentadas com premissas explicitas e ajustaveis.

In [ ]:
import pandas as pd

tn, fp, fn, tp = resultados["matriz_confusao"].ravel()
total_validacao = len(y_valid)

taxa_detecao_buracos = tp / (tp + fn) if (tp + fn) else 0.0  # recall da classe buracos
precisao_alertas = tp / (tp + fp) if (tp + fp) else 0.0
taxa_alertas = (tp + fp) / total_validacao
reducao_triagem = 1.0 - taxa_alertas

# Analise de priorizacao: quantos buracos reais aparecem no topo das probabilidades
probs = resultados["probabilidades"]
y_valid_array = np.asarray(y_valid)
ordem_desc = np.argsort(probs)[::-1]

top_10 = max(1, int(0.10 * total_validacao))
top_20 = max(1, int(0.20 * total_validacao))

buracos_totais = int(np.sum(y_valid_array == MAPA_CLASSES["buracos"]))
buracos_top_10 = int(np.sum(y_valid_array[ordem_desc[:top_10]] == MAPA_CLASSES["buracos"]))
buracos_top_20 = int(np.sum(y_valid_array[ordem_desc[:top_20]] == MAPA_CLASSES["buracos"]))

captura_top_10 = buracos_top_10 / buracos_totais if buracos_totais else 0.0
captura_top_20 = buracos_top_20 / buracos_totais if buracos_totais else 0.0

tabela_operacional = pd.DataFrame(
    [
        ["Acuracia geral", resultados["acuracia"]],
        ["Taxa de deteccao de buracos (recall)", taxa_detecao_buracos],
        ["Precisao dos alertas de buraco", precisao_alertas],
        ["Taxa de imagens sinalizadas para triagem", taxa_alertas],
        ["Reducao potencial de triagem manual", reducao_triagem],
        ["Captura de buracos no Top 10%", captura_top_10],
        ["Captura de buracos no Top 20%", captura_top_20],
    ],
    columns=["Indicador", "Valor"],
)

tabela_operacional["Valor"] = (tabela_operacional["Valor"] * 100).round(2).astype(str) + "%"
tabela_operacional

### Premissas de negocio (editaveis)

Ajuste os valores abaixo para o seu contexto de operacao (prefeitura, concessionaria, frota, etc.).

In [ ]:
# Premissas (exemplo)
minutos_triagem_manual_por_imagem = 2.0
custo_hora_equipe = 120.0  # R$/hora
custo_reparo_precoce = 350.0
custo_reparo_tardio = 950.0

ganho_unitario_reparo = max(0.0, custo_reparo_tardio - custo_reparo_precoce)

# Economia de tempo: imagens nao sinalizadas deixam de ir para triagem imediata
imagens_evitar_triagem = tn + fn
horas_economizadas = (imagens_evitar_triagem * minutos_triagem_manual_por_imagem) / 60.0
economia_tempo_rs = horas_economizadas * custo_hora_equipe

# Economia de manutencao: buracos detectados cedo podem reduzir custo de reparo
economia_manutencao_rs = tp * ganho_unitario_reparo

economia_total_rs = economia_tempo_rs + economia_manutencao_rs

resumo_beneficio = pd.DataFrame(
    [
        ["Imagens de validacao", total_validacao],
        ["Buracos detectados (TP)", tp],
        ["Buracos perdidos (FN)", fn],
        ["Horas estimadas economizadas", round(horas_economizadas, 2)],
        ["Economia estimada de tempo (R$)", round(economia_tempo_rs, 2)],
        ["Economia estimada de manutencao (R$)", round(economia_manutencao_rs, 2)],
        ["Economia total estimada no conjunto de validacao (R$)", round(economia_total_rs, 2)],
    ],
    columns=["Metrica", "Estimativa"],
)
resumo_beneficio

### Leitura executiva para o leitor

- O modelo permite **priorizar as imagens mais criticas** (Top 10% e Top 20%) com base em probabilidade.
- Isso gera **ganho operacional direto** (menos triagem total) e **potencial de economia em manutencao** (deteccao antecipada).
- As premissas podem ser calibradas para refletir o custo real da sua operacao e transformar o estudo em business case.

## 9. Conclusão

Pipeline baseline concluido com sucesso, usando codigo reutilizavel em `src` e dataset local, pronto para evolucao futura (aumento de dados, tuning e novos modelos).